# Analisis de resultados del benchmark

Cada caso corre su `bench.py`, que entrena la red una vez, ejecuta cada backend
en su **propio subproceso** (con su propio venv segun `bench/envs.toml`) y vuelca
un `results.csv`.

**Calidad de la inferencia ML** (atribuida a la capa ML de cada libreria):
- Descomposicion de accuracy: `float_accuracy` (modelo original) ->
  `approx_accuracy` (modelo con la aproximacion FHE pero **en claro**: polinomio
  para la SDK, cuantizacion para Concrete-ML, Chebyshev para Orion) ->
  `accuracy` (cifrado). El salto float->approx es el costo de la *estrategia de
  aproximacion* de la libreria; approx->cifrado es el ruido CKKS residual.
- **Fidelidad predictiva** respecto al modelo en claro (no a las etiquetas):
  `agreement` (fraccion de muestras donde argmax cifrado == argmax del modelo
  original), `output_mae` y `precision_bits` (cercania de los logits cifrados a
  los del modelo original; `precision_bits` es None cuando el error es 0).

**Costo / recursos:**
- `keygen_s`, `compile_s`, y `setup_s = keygen_s + compile_s` (costo unico de
  preparar el modelo, amortizable sobre muchas inferencias).
- `latency_s` (por muestra), `vram_mb` (huella NVML; ya con el piso del GPU
  restado), `vram_alloc_mb` (working set real: contador del pool RMM para la SDK,
  `torch.cuda.memory_allocated` para PyTorch; 0 = no disponible para
  Concrete-ML/Orion), `ram_mb` (pico RSS).

Nota: el playground usa `Square`/`Quad` (x^2), exactamente representable en FHE,
asi que su `approx_accuracy == float_accuracy` (sin costo de aproximacion); la
brecha solo aparece con activaciones reales (ReLU) en mlp/cnn. La cuantizacion de
Concrete-ML si introduce una brecha incluso con x^2.

Este notebook solo **lee** los CSV. Para regenerar datos corre el `bench.py` del caso.

In [ ]:
import glob
import pandas as pd

paths = sorted(glob.glob('*/results.csv'))
frames = [pd.read_csv(p) for p in paths]
df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print('casos:', paths)
df

## Figuras por metrica

La latencia se grafica en escala logaritmica porque el plaintext y los
backends cifrados difieren en varios ordenes de magnitud.

In [ ]:
import matplotlib.pyplot as plt

acc_cols = [c for c in ['float_accuracy', 'approx_accuracy', 'accuracy'] if c in df]
if not df.empty and acc_cols:
    ax = df.set_index('backend')[acc_cols].plot(
        kind='bar', figsize=(7, 4),
        title='accuracy decomposition (float -> approx -> encrypted)')
    ax.set_ylabel('accuracy')
    plt.tight_layout()
    plt.show()

LOG_METRICS = {'keygen_s', 'compile_s', 'setup_s', 'latency_s'}
METRICS = ['setup_s', 'keygen_s', 'compile_s', 'latency_s',
           'agreement', 'output_mae', 'precision_bits',
           'vram_mb', 'vram_alloc_mb', 'ram_mb']
for metric in METRICS:
    if df.empty or metric not in df:
        continue
    pivot = df.pivot(index='backend', columns='case', values=metric)
    pivot.plot(kind='bar', logy=metric in LOG_METRICS, figsize=(6, 4), title=metric)
    plt.tight_layout()
    plt.show()

## Perfil por capa de la SDK (solo SDK)

`profile_sdk.csv` (lo genera `profile_sdk.py`, que corre despues de las metricas
de comparacion) mide, capa por capa, el tiempo (sincronizado en GPU) y la memoria
*real* del pool RMM: `mem_delta_mb` es lo que cada capa agrega — puede ser
negativo, p.ej. un rescale libera un primo — y `mem_after_mb` es el total vivo
acumulado. Solo la SDK lo soporta; Concrete-ML y Orion compilan a un circuito
fusionado sin ganchos por capa.

In [ ]:
import glob
import pandas as pd
import matplotlib.pyplot as plt

for path in sorted(glob.glob('*/profile_sdk.csv')):
    case = path.split('/')[0]
    p = pd.read_csv(path)
    label = p['layer_idx'].astype(str) + ':' + p['layer_name']
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].bar(label, p['time_s'] * 1000)
    ax[0].set_title(f'{case}: SDK per-layer time (ms)')
    ax[0].tick_params(axis='x', rotation=45)
    ax[1].plot(label, p['mem_after_mb'], marker='o', label='live')
    ax[1].bar(label, p['mem_delta_mb'], alpha=0.4, label='delta')
    ax[1].set_title(f'{case}: SDK per-layer memory (MB)')
    ax[1].tick_params(axis='x', rotation=45)
    ax[1].legend()
    plt.tight_layout()
    plt.show()